# Topo-Brain: Topology-Preserving 3T-to-7T MRI Enhancement

## Complete Training, Inference & Evaluation Pipeline

This notebook provides a unified, end-to-end pipeline for:
1. **Configuration & Setup** - Load config, verify GPU, check data
2. **Data Exploration** - Visualize paired 3T/7T volumes and patches
3. **Training** - Full curriculum-based diffusion training with EMA
4. **Checkpoint Analysis** - Find best checkpoint from training
5. **Single-Patch Inference** - Quick inference on center 64^3 crop
6. **Full-Volume Inference** - Tiled inference with overlap stitching
7. **Quantitative Evaluation** - SSIM, PSNR, segmentation metrics
8. **Cross-Subject Evaluation** - Evaluate across all test subjects

### Architecture
- **Model**: Anatomy-Guided U-Net with dual decoder (denoising + segmentation)
- **Diffusion**: Gaussian diffusion with cosine beta schedule (200 timesteps)
- **Losses**: L1 diffusion + L1 pixel + VGG perceptual + Multi-scale topology
- **Training**: 4-stage curriculum with progressive loss introduction

---
## 1. Configuration & Setup

In [ ]:
import os
import sys
import yaml
import csv
import time
import logging
import numpy as np
import nibabel as nib
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import display, HTML

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

# Project root setup
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Import Topo-Brain modules
from src.model import AnatomyGuidedUNet
from src.diffusion import GaussianDiffusion
from src.synthesis_dataset import (
    create_synthesis_dataloaders, load_pairs_manifest,
    PatchConfig, SplitConfig, PairedPatchDataset, SubjectSplitter
)
from src.utils import setup_logging, TensorBoardLogger

print("All Topo-Brain modules imported successfully.")

In [ ]:
# ============================================================
# USER CONFIGURATION - Edit these paths for your setup
# ============================================================

CONFIG_PATH = PROJECT_ROOT / "configs" / "train_diffusion.yaml"

# Set these to your actual data directories
DATA_ROOT = None       # e.g., Path(r"D:\data\topobrain-preproc") or None if CSV has absolute paths
MASKS_ROOT = None      # e.g., Path(r"D:\data\seg-masks") or None

# Output directory for training checkpoints and logs
OUTPUT_DIR = PROJECT_ROOT / "output"

# Resume from checkpoint (set to path or None to train from scratch)
RESUME_CHECKPOINT = None  # e.g., OUTPUT_DIR / "checkpoint_latest.pt"

# ============================================================

# Load config
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Batch size: {config['dataset']['batch_size']}")
print(f"  Patch size: {config['dataset']['patch_size']}")
print(f"  Timesteps: {config['diffusion']['timesteps']}")
print(f"  Beta schedule: {config['diffusion']['beta_schedule']}")
print(f"  Total iterations: {config['training']['n_iters']}")
print(f"  Learning rate: {config['training']['lr']}")
print(f"  Features: {config['model']['features']}")
print(f"  Attention: {config['model']['use_attention']}")
print(f"  Loss weights: pixel={config['loss_weights']['lambda_pixel']}, "
      f"percep={config['loss_weights']['lambda_percep']}, "
      f"topo={config['loss_weights']['lambda_topo']}")

---
## 2. Data Exploration

In [ ]:
# Load pairs manifest
pairs_csv = PROJECT_ROOT / config['dataset']['pairs_csv']
pairs = load_pairs_manifest(pairs_csv)

# Prepend data root if provided
if DATA_ROOT:
    for p in pairs:
        if 'input_3t' in p: p['input_3t'] = str(Path(DATA_ROOT) / p['input_3t'])
        if 'target_7t' in p: p['target_7t'] = str(Path(DATA_ROOT) / p['target_7t'])
if MASKS_ROOT:
    for p in pairs:
        if 'seg' in p and p['seg']: p['seg'] = str(Path(MASKS_ROOT) / p['seg'])

print(f"Loaded {len(pairs)} subject pairs:")
for p in pairs:
    subj = p.get('subject', '?')
    inp_exists = Path(p['input_3t']).exists() if p.get('input_3t') else False
    tgt_exists = Path(p['target_7t']).exists() if p.get('target_7t') else False
    seg_exists = Path(p['seg']).exists() if p.get('seg') else False
    status = lambda x: 'OK' if x else 'MISSING'
    print(f"  {subj}: input={status(inp_exists)}, target={status(tgt_exists)}, seg={status(seg_exists)}")

In [ ]:
# Visualize a sample 3T/7T pair
def visualize_pair(pair, slice_frac=0.5):
    """Visualize a 3T/7T pair at the given fractional slice position."""
    inp_path, tgt_path = pair['input_3t'], pair['target_7t']
    subj = pair.get('subject', '?')
    
    if not Path(inp_path).exists():
        print(f"Skipping {subj}: input not found")
        return
    
    inp = nib.load(inp_path).get_fdata().astype(np.float32)
    tgt = nib.load(tgt_path).get_fdata().astype(np.float32) if Path(tgt_path).exists() else None
    
    D, H, W = inp.shape
    sl = int(D * slice_frac)
    
    ncols = 3 if tgt is not None else 1
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
    if ncols == 1: axes = [axes]
    
    vmin, vmax = -1.0, 1.0
    
    axes[0].imshow(np.rot90(inp[sl]), cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].set_title(f'3T Input (slice {sl})', fontsize=13)
    axes[0].axis('off')
    
    if tgt is not None:
        axes[1].imshow(np.rot90(tgt[sl]), cmap='gray', vmin=vmin, vmax=vmax)
        axes[1].set_title(f'7T Target (slice {sl})', fontsize=13)
        axes[1].axis('off')
        
        diff = np.abs(tgt[sl] - inp[sl])
        im = axes[2].imshow(np.rot90(diff), cmap='hot', vmin=0, vmax=0.5)
        axes[2].set_title('|7T - 3T| Difference', fontsize=13)
        axes[2].axis('off')
        plt.colorbar(im, ax=axes[2], fraction=0.046)
    
    plt.suptitle(f'Subject: {subj}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"  Volume shape: {inp.shape}")
    print(f"  3T range: [{inp.min():.3f}, {inp.max():.3f}]")
    if tgt is not None:
        print(f"  7T range: [{tgt.min():.3f}, {tgt.max():.3f}]")

# Show first 2 subjects
for p in pairs[:2]:
    visualize_pair(p)

In [ ]:
# Visualize training patches from the dataset
dataset_cfg = config['dataset']
patch_cfg = PatchConfig(
    patch_size=tuple(dataset_cfg.get('patch_size', (64, 64, 64))),
    patches_per_volume=int(dataset_cfg.get('patches_per_volume', 32)),
    min_brain_fraction=float(dataset_cfg.get('min_brain_fraction', 0.1)),
    use_t2=bool(dataset_cfg.get('use_t2', False)),
    seed=int(dataset_cfg.get('seed', 42)),
)

# Create a small dataset to visualize patches
viz_dataset = PairedPatchDataset(pairs[:2], config=patch_cfg, augment=False, cache_volumes=True)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i in range(4):
    sample = viz_dataset[i]
    inp_patch = sample['input'][0].numpy()   # [D, H, W]
    tgt_patch = sample['target'][0].numpy()
    sl = inp_patch.shape[0] // 2
    
    axes[0, i].imshow(np.rot90(inp_patch[sl]), cmap='gray', vmin=-1, vmax=1)
    axes[0, i].set_title(f'3T Patch {i} (z={sl})', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.rot90(tgt_patch[sl]), cmap='gray', vmin=-1, vmax=1)
    axes[1, i].set_title(f'7T Patch {i} (z={sl})', fontsize=11)
    axes[1, i].axis('off')

plt.suptitle('Training Patches (64x64x64)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

del viz_dataset  # Free memory

---
## 3. Training

### Curriculum Overview
| Stage | Steps | Active Losses |
|-------|-------|---------------|
| 1 | 0 - 3k | Diffusion + 0.5x Pixel |
| 2 | 3k - 12k | Diffusion + Full Pixel |
| 3 | 12k - 30k | + Perceptual (warmup 8k steps) |
| 4 | 30k - 150k | + Topology (warmup 25k steps) |

In [ ]:
# ============================================================
# Data Loaders
# ============================================================

split_cfg = SplitConfig(
    n_folds=int(dataset_cfg.get('n_folds', 10)),
    val_fold=int(dataset_cfg.get('val_fold', 0)),
    test_fold=int(dataset_cfg.get('test_fold', 1)),
    use_loocv=bool(dataset_cfg.get('use_loocv', True)),
    seed=int(dataset_cfg.get('seed', 42)),
)

train_loader, val_loader, test_loader = create_synthesis_dataloaders(
    pairs,
    config=patch_cfg,
    split_config=split_cfg,
    batch_size=int(dataset_cfg.get('batch_size', 4)),
    num_workers=int(dataset_cfg.get('num_workers', 4)),
    val_fold=int(dataset_cfg.get('val_fold', 0)),
)

def cycle(dl):
    """Infinite dataloader iterator."""
    while True:
        for data in dl:
            yield data

train_iter = cycle(train_loader)
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# ============================================================
# Model, Diffusion, Optimizer, EMA
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

use_t2 = config['dataset'].get('use_t2', False)
in_channels = 2 if use_t2 else 1

model = AnatomyGuidedUNet(
    in_channels=in_channels,
    cond_channels=1,
    out_channels=1,
    num_classes=config['model'].get('num_classes', 4),
    features=tuple(config['model'].get('features', (32, 64, 128, 256))),
    use_attention=config['model'].get('use_attention', False),
).to(device)

diffusion = GaussianDiffusion(
    model,
    timesteps=config['diffusion'].get('timesteps', 1000),
    beta_schedule=config['diffusion'].get('beta_schedule', 'cosine'),
).to(device)

# EMA model
class EMA:
    def __init__(self, beta):
        self.beta = beta
        self.step = 0
    def update_model_average(self, ma_model, current_model):
        for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
            old_weight, up_weight = ma_params.data, current_params.data
            ma_params.data = old_weight * self.beta + (1 - self.beta) * up_weight
    def step_ema(self, ema_model, model, step_start_ema=2000):
        if self.step < step_start_ema:
            self.step += 1
            return
        self.update_model_average(ema_model, model)
        self.step += 1

ema = EMA(config['training'].get('ema_decay', 0.9999))
ema_model = AnatomyGuidedUNet(
    in_channels=in_channels, cond_channels=1, out_channels=1,
    num_classes=config['model'].get('num_classes', 4),
    features=tuple(config['model'].get('features', (32, 64, 128, 256))),
    use_attention=config['model'].get('use_attention', False),
).to(device)
ema_model.load_state_dict(model.state_dict())
ema_model.requires_grad_(False)

# Optimizer
lr = float(config['training']['lr'])
optimizer = optim.AdamW(
    model.parameters(), lr=lr,
    weight_decay=float(config['training'].get('weight_decay', 1e-4)),
    eps=1e-5,
)

# AMP
use_amp = bool(config['training'].get('use_amp', True))
scaler = GradScaler(enabled=use_amp, init_scale=2**12)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,} total, {n_trainable:,} trainable")
print(f"Device: {device}")
print(f"AMP: {use_amp}")

In [ ]:
# ============================================================
# Resume from checkpoint (optional)
# ============================================================

start_step = 0

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Resuming from: {RESUME_CHECKPOINT}")
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
    
    model.load_state_dict(checkpoint['model'], strict=False)
    if 'ema' in checkpoint:
        ema_model.load_state_dict(checkpoint['ema'], strict=False)
    if 'optimizer' in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint['optimizer'])
        except Exception as e:
            print(f"Could not load optimizer state: {e}")
    if 'scaler' in checkpoint and use_amp:
        try:
            scaler.load_state_dict(checkpoint['scaler'])
        except Exception as e:
            print(f"Could not load scaler state: {e}")
    if 'step' in checkpoint:
        start_step = checkpoint['step'] + 1
    
    print(f"Resumed at step {start_step}")
    del checkpoint
    torch.cuda.empty_cache()
else:
    print("Training from scratch.")

In [ ]:
# ============================================================
# Training Loop
# ============================================================

n_iters = config['training']['n_iters']
grad_clip = config['training'].get('grad_clip', 1.0)
save_freq = config['training']['save_freq']
log_freq = config['training'].get('log_freq', 50)
num_classes = config['model'].get('num_classes', 4)

# Curriculum stages
stages = config.get('stages', {})
stage1_end = stages.get('stage1_end', 10000)
stage2_end = stages.get('stage2_end', 50000)
stage3_end = stages.get('stage3_end', 100000)

# Loss weights
loss_cfg = config.get('loss_weights', {})
final_lambda_pixel = loss_cfg.get('lambda_pixel', 1.0)
final_lambda_percep = loss_cfg.get('lambda_percep', 0.25)
final_lambda_topo = loss_cfg.get('lambda_topo', 0.2)
topo_warmup_steps = loss_cfg.get('topo_warmup_steps', 25000)
percep_warmup_steps = loss_cfg.get('percep_warmup_steps', 8000)

# Logging
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log_dir = OUTPUT_DIR / "logs" / time.strftime("%Y%m%d-%H%M%S")
log_dir.mkdir(parents=True, exist_ok=True)
tb_logger = TensorBoardLogger(log_dir)

# Training history for plotting
history = {
    'steps': [], 'loss_total': [], 'loss_diff': [],
    'loss_pixel': [], 'loss_percep': [], 'loss_topo': [],
}

skipped_steps = 0
attempted_steps = 0

print(f"Starting training from step {start_step} to {n_iters}")
print(f"Curriculum: Stage1<{stage1_end}, Stage2<{stage2_end}, Stage3<{stage3_end}, Stage4>{stage3_end}")
print(f"Checkpoints saved every {save_freq} steps to: {OUTPUT_DIR}")
print(f"TensorBoard logs: {log_dir}")
print("="*70)

pbar = tqdm(range(start_step, n_iters), initial=start_step, total=n_iters, desc="Training")

for step in pbar:
    attempted_steps += 1
    optimizer.zero_grad()
    
    # --- Get batch ---
    batch = next(train_iter)
    x_start = batch['target'].to(device)
    cond = batch['input'].to(device)
    seg_target = batch['seg'].to(device)
    if len(seg_target.shape) == 5:
        seg_target = seg_target.squeeze(1)
    
    # Clamp invalid labels
    invalid_mask = (seg_target < 0) | (seg_target >= num_classes)
    if invalid_mask.any():
        seg_target = seg_target.clone()
        seg_target[invalid_mask] = 0
    
    # --- Curriculum loss weights ---
    if step < stage1_end:
        lambda_pixel = final_lambda_pixel * 0.5
        lambda_percep, lambda_topo = 0.0, 0.0
    elif step < stage2_end:
        lambda_pixel = final_lambda_pixel
        lambda_percep, lambda_topo = 0.0, 0.0
    elif step < stage3_end:
        steps_into_s3 = step - stage2_end
        lambda_pixel = final_lambda_pixel
        lambda_percep = final_lambda_percep * min(1.0, steps_into_s3 / percep_warmup_steps)
        lambda_topo = 0.0
    else:
        steps_into_s4 = step - stage3_end
        lambda_pixel = final_lambda_pixel
        lambda_percep = final_lambda_percep
        lambda_topo = final_lambda_topo * min(1.0, steps_into_s4 / topo_warmup_steps)
    
    # --- Forward pass ---
    with autocast(device_type=device.type, enabled=use_amp):
        loss_dict = diffusion(
            x_start, cond, seg_target,
            lambda_pixel=lambda_pixel,
            lambda_percep=lambda_percep,
            lambda_topo=lambda_topo,
        )
    
    # Non-finite loss guard
    non_finite = [k for k, v in loss_dict.items()
                  if isinstance(v, torch.Tensor) and not torch.isfinite(v).all()]
    if non_finite:
        skipped_steps += 1
        optimizer.zero_grad()
        scaler.update()
        continue
    
    # --- Backward pass ---
    scaler.scale(loss_dict['loss']).backward()
    scaler.unscale_(optimizer)
    
    # NaN gradient guard
    has_nan_grad = any(
        torch.isnan(p.grad).any() or torch.isinf(p.grad).any()
        for p in model.parameters() if p.grad is not None
    )
    if has_nan_grad:
        skipped_steps += 1
        optimizer.zero_grad()
        scaler.update()
        continue
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    
    # LR decay
    lr_decay_step = config['training'].get('lr_decay_step', 0)
    if lr_decay_step > 0 and step == lr_decay_step:
        new_lr = config['training']['lr'] * config['training'].get('lr_decay_factor', 0.5)
        for pg in optimizer.param_groups:
            pg['lr'] = new_lr
        print(f"\nLR decayed to {new_lr}")
    
    scaler.step(optimizer)
    scaler.update()
    ema.step_ema(ema_model, model)
    
    # --- Logging ---
    tb_logger.log_scalar('Loss/Total', loss_dict['loss'].item(), step)
    tb_logger.log_scalar('Loss/Diff', loss_dict['loss_diff'].item(), step)
    tb_logger.log_scalar('Loss/Pixel', loss_dict['loss_pixel'].item(), step)
    tb_logger.log_scalar('Loss/Perceptual', loss_dict['loss_vgg'].item(), step)
    tb_logger.log_scalar('Loss/Topo', loss_dict['loss_topo'].item(), step)
    
    if step % log_freq == 0:
        history['steps'].append(step)
        history['loss_total'].append(loss_dict['loss'].item())
        history['loss_diff'].append(loss_dict['loss_diff'].item())
        history['loss_pixel'].append(loss_dict['loss_pixel'].item())
        history['loss_percep'].append(loss_dict['loss_vgg'].item())
        history['loss_topo'].append(loss_dict['loss_topo'].item())
        
        skip_ratio = skipped_steps / attempted_steps if attempted_steps > 0 else 0
        pbar.set_postfix({
            'total': f"{loss_dict['loss'].item():.4f}",
            'diff': f"{loss_dict['loss_diff'].item():.4f}",
            'skip%': f"{100*skip_ratio:.1f}",
        })
    
    # --- Checkpoint ---
    if step > 0 and step % save_freq == 0:
        step_dir = OUTPUT_DIR / f"checkpoint_{step}"
        step_dir.mkdir(parents=True, exist_ok=True)
        
        ckpt_data = {
            'step': step,
            'model': model.state_dict(),
            'ema': ema_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict(),
            'config': config,
        }
        
        torch.save(ckpt_data, step_dir / f"checkpoint_{step}.pt")
        torch.save(ckpt_data, OUTPUT_DIR / "checkpoint_latest.pt")
        print(f"\nCheckpoint saved: step {step}")

print(f"\nTraining complete. Skipped: {skipped_steps}/{attempted_steps} ({100*skipped_steps/max(1,attempted_steps):.1f}%)")
tb_logger.close()

In [ ]:
# ============================================================
# Plot Training Curves
# ============================================================

if history['steps']:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    axes[0, 0].plot(history['steps'], history['loss_total'], 'b-', alpha=0.7)
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(history['steps'], history['loss_diff'], 'r-', alpha=0.7, label='Diffusion')
    axes[0, 1].plot(history['steps'], history['loss_pixel'], 'g-', alpha=0.7, label='Pixel')
    axes[0, 1].set_title('Diffusion & Pixel Loss')
    axes[0, 1].legend()
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(history['steps'], history['loss_percep'], 'm-', alpha=0.7)
    axes[1, 0].set_title('Perceptual Loss')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(history['steps'], history['loss_topo'], 'c-', alpha=0.7)
    axes[1, 1].set_title('Topology Loss')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Mark curriculum stage boundaries
    for ax in axes.flat:
        for s, label in [(stage1_end, 'S1'), (stage2_end, 'S2'), (stage3_end, 'S3')]:
            ax.axvline(x=s, color='gray', linestyle='--', alpha=0.5)
            ax.text(s, ax.get_ylim()[1], label, fontsize=8, ha='center', va='bottom')
    
    plt.suptitle('Training Loss Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No training history to plot.")

---
## 4. Checkpoint Analysis

Find and load the best checkpoint for inference.

In [ ]:
# List all available checkpoints
ckpt_dirs = sorted(OUTPUT_DIR.glob('checkpoint_*/checkpoint_*.pt'))

print(f"Found {len(ckpt_dirs)} checkpoints:")
for ckpt in ckpt_dirs:
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  {ckpt.name} ({size_mb:.0f} MB)")

# Also check for latest
latest = OUTPUT_DIR / 'checkpoint_latest.pt'
if latest.exists():
    print(f"\nLatest checkpoint available: {latest}")

In [ ]:
# ============================================================
# SELECT CHECKPOINT FOR INFERENCE
# ============================================================

# Option 1: Use latest
INFERENCE_CHECKPOINT = OUTPUT_DIR / 'checkpoint_latest.pt'

# Option 2: Use specific step
# INFERENCE_CHECKPOINT = OUTPUT_DIR / 'checkpoint_75000' / 'checkpoint_75000.pt'

print(f"Selected checkpoint: {INFERENCE_CHECKPOINT}")
assert INFERENCE_CHECKPOINT.exists(), f"Checkpoint not found: {INFERENCE_CHECKPOINT}"

---
## 5. Single-Patch Inference (Quick)

Fast inference on a center 64^3 crop. Good for quick quality checks.

In [ ]:
# ============================================================
# Load Model for Inference
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint
ckpt = torch.load(INFERENCE_CHECKPOINT, map_location=device)
ckpt_config = ckpt.get('config', config)

# Build model from checkpoint config
inf_use_t2 = ckpt_config['dataset'].get('use_t2', False)
inf_in_ch = 2 if inf_use_t2 else 1

inf_model = AnatomyGuidedUNet(
    in_channels=inf_in_ch,
    cond_channels=1,
    out_channels=1,
    num_classes=ckpt_config['model'].get('num_classes', 4),
    features=tuple(ckpt_config['model'].get('features', (32, 64, 128, 256))),
    use_attention=ckpt_config['model'].get('use_attention', False),
).to(device)

inf_diffusion = GaussianDiffusion(
    inf_model,
    timesteps=ckpt_config['diffusion'].get('timesteps', 1000),
    beta_schedule=ckpt_config['diffusion'].get('beta_schedule', 'cosine'),
).to(device)

# Load EMA weights (preferred) or model weights
if 'ema' in ckpt:
    inf_model.load_state_dict(ckpt['ema'])
    print("Loaded EMA weights")
else:
    inf_model.load_state_dict(ckpt['model'])
    print("Loaded model weights")

inf_model.eval()
step_loaded = ckpt.get('step', '?')
print(f"Model loaded from step {step_loaded}")
del ckpt
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Quick Patch Inference on a Test Subject
# ============================================================
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Pick a test subject
test_subject = pairs[int(dataset_cfg.get('test_fold', 1))]  # Subject used as test fold
print(f"Test subject: {test_subject['subject']}")

inp_path = test_subject['input_3t']
tgt_path = test_subject['target_7t']

# Load volumes
inp_nii = nib.load(inp_path)
input_vol = np.clip(inp_nii.get_fdata().astype(np.float32), -1.0, 1.0)
target_vol = np.clip(nib.load(tgt_path).get_fdata().astype(np.float32), -1.0, 1.0)

# Center crop 64^3
D, H, W = input_vol.shape
cd, ch, cw = D//2, H//2, W//2
rad = 32

inp_crop = input_vol[cd-rad:cd+rad, ch-rad:ch+rad, cw-rad:cw+rad]
tgt_crop = target_vol[cd-rad:cd+rad, ch-rad:ch+rad, cw-rad:cw+rad]

# To tensor
inp_tensor = torch.from_numpy(inp_crop).float().unsqueeze(0).unsqueeze(0).to(device)

print(f"Volume shape: {input_vol.shape}")
print(f"Crop shape: {inp_crop.shape}")
print("Running diffusion sampling...")

# Run inference
with torch.no_grad():
    out_shape = (1, 1, 64, 64, 64)
    generated, pred_seg = inf_diffusion.p_sample_loop(
        conditioning=inp_tensor,
        shape=out_shape,
        return_all=True,
    )

gen_np = generated.cpu().numpy().squeeze()
gen_np = np.clip(gen_np, -1.0, 1.0)

# Compute metrics
gen_01 = (gen_np + 1.0) / 2.0
tgt_01 = (tgt_crop + 1.0) / 2.0

ssim_val = ssim(tgt_01, gen_01, data_range=1.0)
psnr_val = psnr(tgt_01, gen_01, data_range=1.0)

print(f"\nPatch Metrics (center 64^3):")
print(f"  SSIM: {ssim_val:.4f}")
print(f"  PSNR: {psnr_val:.2f} dB")

In [ ]:
# Visualize patch inference results
sl = 32  # Middle of 64^3 patch

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
vmin, vmax = -1.0, 1.0

axes[0].imshow(np.rot90(inp_crop[sl]), cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('3T Input', fontsize=14)
axes[0].axis('off')

axes[1].imshow(np.rot90(gen_np[sl]), cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title(f'Predicted 7T\nSSIM={ssim_val:.4f}, PSNR={psnr_val:.1f}dB', fontsize=14)
axes[1].axis('off')

axes[2].imshow(np.rot90(tgt_crop[sl]), cmap='gray', vmin=vmin, vmax=vmax)
axes[2].set_title('Ground Truth 7T', fontsize=14)
axes[2].axis('off')

error = np.abs(gen_np[sl] - tgt_crop[sl])
im = axes[3].imshow(np.rot90(error), cmap='hot', vmin=0, vmax=0.5)
axes[3].set_title('Absolute Error', fontsize=14)
axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)

plt.suptitle(f'Patch Inference - {test_subject["subject"]} (step {step_loaded})', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Segmentation visualization
if pred_seg is not None:
    seg_map = torch.argmax(pred_seg, dim=1).cpu().numpy().squeeze()
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(np.rot90(seg_map[sl]), cmap='viridis', vmin=0, vmax=3)
    ax.set_title('Predicted Segmentation\n(0=BG, 1=CSF, 2=GM, 3=WM)', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 6. Full-Volume Inference (Tiled)

Runs overlapping 64^3 patches across the entire volume with Tukey window blending.

In [ ]:
# ============================================================
# Tiled Full-Volume Inference
# ============================================================

def _tukey_window_1d(n, alpha=0.5):
    """1-D Tukey (tapered-cosine) window."""
    if alpha <= 0:
        return np.ones(n)
    if alpha >= 1:
        return np.hanning(n)
    w = np.ones(n)
    taper = int(alpha * n / 2)
    w[:taper] = 0.5 * (1 - np.cos(np.pi * np.arange(taper) / taper))
    w[-taper:] = 0.5 * (1 - np.cos(np.pi * np.arange(taper, 0, -1) / taper))
    return w


def tiled_inference(diffusion_model, input_vol, device, patch_size=64, overlap=32):
    """
    Run diffusion inference on overlapping tiles and stitch.
    
    Returns:
        output_vol: [D, H, W] predicted 7T volume
        seg_vol: [D, H, W] predicted segmentation (argmax)
    """
    D, H, W = input_vol.shape
    stride = patch_size - overlap
    n_classes = 4
    
    output_acc = np.zeros((D, H, W), dtype=np.float64)
    seg_acc = np.zeros((n_classes, D, H, W), dtype=np.float64)
    weight_acc = np.zeros((D, H, W), dtype=np.float64)
    
    alpha = overlap / patch_size
    w1d = _tukey_window_1d(patch_size, alpha=alpha)
    window = w1d[None, None, :] * w1d[None, :, None] * w1d[:, None, None]
    window = window.astype(np.float64)
    
    def get_starts(length, ps, st):
        starts = list(range(0, length - ps + 1, st))
        if starts and starts[-1] + ps < length:
            starts.append(length - ps)
        if not starts:
            starts = [max(0, (length - ps) // 2)]
        return starts
    
    d_starts = get_starts(D, patch_size, stride)
    h_starts = get_starts(H, patch_size, stride)
    w_starts = get_starts(W, patch_size, stride)
    
    total = len(d_starts) * len(h_starts) * len(w_starts)
    pbar = tqdm(total=total, desc='Tiled inference', unit='patch')
    
    for ds in d_starts:
        for hs in h_starts:
            for ws in w_starts:
                patch = input_vol[ds:ds+patch_size, hs:hs+patch_size, ws:ws+patch_size]
                actual_shape = patch.shape
                
                if actual_shape != (patch_size, patch_size, patch_size):
                    padded = np.zeros((patch_size, patch_size, patch_size), dtype=np.float32)
                    padded[:actual_shape[0], :actual_shape[1], :actual_shape[2]] = patch
                    patch = padded
                
                inp = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    pred, seg = diffusion_model.p_sample_loop(
                        conditioning=inp, shape=inp.shape, return_all=True
                    )
                
                pred_np = pred.cpu().numpy().squeeze()
                pred_np = pred_np[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                win = window[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                output_acc[ds:ds+actual_shape[0], hs:hs+actual_shape[1], ws:ws+actual_shape[2]] += pred_np * win
                weight_acc[ds:ds+actual_shape[0], hs:hs+actual_shape[1], ws:ws+actual_shape[2]] += win
                
                if seg is not None:
                    seg_np = seg.cpu().numpy().squeeze()
                    seg_np = seg_np[:, :actual_shape[0], :actual_shape[1], :actual_shape[2]]
                    for c in range(seg_np.shape[0]):
                        seg_acc[c, ds:ds+actual_shape[0], hs:hs+actual_shape[1], ws:ws+actual_shape[2]] += seg_np[c] * win
                
                pbar.update(1)
    
    pbar.close()
    
    mask = weight_acc > 0
    output_vol = np.zeros_like(output_acc, dtype=np.float32)
    output_vol[mask] = (output_acc[mask] / weight_acc[mask]).astype(np.float32)
    
    for c in range(seg_acc.shape[0]):
        seg_acc[c][mask] /= weight_acc[mask]
    seg_vol = np.argmax(seg_acc, axis=0).astype(np.uint8)
    
    return output_vol, seg_vol

print("Tiled inference function ready.")

In [ ]:
# ============================================================
# Run Full-Volume Inference
# ============================================================

print(f"Running full-volume inference on {test_subject['subject']}...")
print(f"Volume shape: {input_vol.shape}")
print(f"Patch size: 64, Overlap: 32")

pred_vol, seg_vol = tiled_inference(
    inf_diffusion, input_vol, device,
    patch_size=64, overlap=32,
)

print(f"Output shape: {pred_vol.shape}")
print(f"Segmentation shape: {seg_vol.shape}")
print(f"Predicted range: [{pred_vol.min():.3f}, {pred_vol.max():.3f}]")

In [ ]:
# Save outputs as NIfTI
results_dir = OUTPUT_DIR / 'results' / test_subject['subject']
results_dir.mkdir(parents=True, exist_ok=True)

affine = inp_nii.affine

nib.save(nib.Nifti1Image(pred_vol, affine), results_dir / 'predicted_7T.nii.gz')
nib.save(nib.Nifti1Image(seg_vol.astype(np.float32), affine), results_dir / 'predicted_seg.nii.gz')

print(f"Saved to: {results_dir}")
print(f"  predicted_7T.nii.gz")
print(f"  predicted_seg.nii.gz")

---
## 7. Quantitative Evaluation

Comprehensive metrics on the full predicted volume.

In [ ]:
# ============================================================
# Compute Full-Volume Metrics
# ============================================================
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

def compute_metrics(pred_np, target_np):
    """Compute SSIM and PSNR. Both arrays expected in [-1, 1]."""
    pred_np = np.clip(pred_np, -1.0, 1.0)
    target_np = np.clip(target_np, -1.0, 1.0)
    pred_01 = (pred_np + 1.0) / 2.0
    tgt_01 = (target_np + 1.0) / 2.0
    ssim_val = ssim(tgt_01, pred_01, data_range=1.0)
    psnr_val = psnr(tgt_01, pred_01, data_range=1.0)
    return ssim_val, psnr_val

def compute_l1_l2(pred_np, target_np):
    """Compute L1 and L2 distance."""
    l1 = np.mean(np.abs(pred_np - target_np))
    l2 = np.sqrt(np.mean((pred_np - target_np) ** 2))
    return l1, l2

# Brain mask for masked metrics
brain_mask = input_vol > -0.95
print(f"Brain mask: {brain_mask.sum():,} voxels ({100*brain_mask.mean():.1f}% of volume)")

# 1) Full-volume masked metrics (primary)
pred_masked = pred_vol.copy()
pred_masked[~brain_mask] = target_vol[~brain_mask]  # Replace BG with GT

ssim_masked, psnr_masked = compute_metrics(pred_masked, target_vol)
l1_masked, l2_masked = compute_l1_l2(pred_masked[brain_mask], target_vol[brain_mask])

# 2) Brain bounding box metrics
coords = np.argwhere(brain_mask)
lo = coords.min(axis=0)
hi = coords.max(axis=0) + 1
pred_bb = pred_masked[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
tgt_bb = target_vol[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
ssim_brain, psnr_brain = compute_metrics(pred_bb, tgt_bb)

# 3) Center 64^3 crop metrics
if min(D, H, W) >= 64:
    crop_pred = pred_masked[cd-rad:cd+rad, ch-rad:ch+rad, cw-rad:cw+rad]
    crop_tgt = target_vol[cd-rad:cd+rad, ch-rad:ch+rad, cw-rad:cw+rad]
    ssim_crop, psnr_crop = compute_metrics(crop_pred, crop_tgt)
else:
    ssim_crop, psnr_crop = float('nan'), float('nan')

# Print results
print("\n" + "="*60)
print(f"METRICS - {test_subject['subject']} (step {step_loaded})")
print("="*60)
print(f"\n{'Metric':<25} {'Masked':>10} {'Brain-BB':>10} {'Center64':>10}")
print("-"*55)
print(f"{'SSIM':<25} {ssim_masked:>10.4f} {ssim_brain:>10.4f} {ssim_crop:>10.4f}")
print(f"{'PSNR (dB)':<25} {psnr_masked:>10.2f} {psnr_brain:>10.2f} {psnr_crop:>10.2f}")
print(f"{'L1 (brain only)':<25} {l1_masked:>10.4f}")
print(f"{'L2 (brain only)':<25} {l2_masked:>10.4f}")

# Segmentation volumes
print(f"\nTissue Segmentation Volumes (predicted):")
labels = {0: 'Background', 1: 'CSF', 2: 'Gray Matter', 3: 'White Matter'}
for cls_id, name in labels.items():
    count = (seg_vol == cls_id).sum()
    pct = 100 * count / seg_vol.size
    print(f"  {name}: {count:>10,} voxels ({pct:.1f}%)")

In [ ]:
# ============================================================
# Multi-View Visualization
# ============================================================

def show_multi_view(input_vol, pred_vol, target_vol, title=""):
    """Show axial, coronal, sagittal views at center."""
    D, H, W = input_vol.shape
    
    views = {
        'Axial': (lambda v, s: v[s, :, :], D // 2),
        'Coronal': (lambda v, s: v[:, s, :], H // 2),
        'Sagittal': (lambda v, s: v[:, :, s], W // 2),
    }
    
    vmin = min(input_vol.min(), pred_vol.min(), target_vol.min())
    vmax = max(input_vol.max(), pred_vol.max(), target_vol.max())
    
    fig, axes = plt.subplots(3, 4, figsize=(24, 18))
    
    for row, (view_name, (func, pos)) in enumerate(views.items()):
        inp_sl = np.rot90(func(input_vol, pos))
        pred_sl = np.rot90(func(pred_vol, pos))
        tgt_sl = np.rot90(func(target_vol, pos))
        err_sl = np.abs(pred_sl - tgt_sl)
        
        axes[row, 0].imshow(inp_sl, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row, 0].set_title(f'{view_name} - 3T Input' if row == 0 else '')
        axes[row, 0].set_ylabel(view_name, fontsize=13, fontweight='bold')
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(pred_sl, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row, 1].set_title('Predicted 7T' if row == 0 else '')
        axes[row, 1].axis('off')
        
        axes[row, 2].imshow(tgt_sl, cmap='gray', vmin=vmin, vmax=vmax)
        axes[row, 2].set_title('Target 7T' if row == 0 else '')
        axes[row, 2].axis('off')
        
        im = axes[row, 3].imshow(err_sl, cmap='hot', vmin=0, vmax=0.5)
        axes[row, 3].set_title('Abs Error' if row == 0 else '')
        axes[row, 3].axis('off')
        plt.colorbar(im, ax=axes[row, 3], fraction=0.046)
    
    plt.suptitle(title or 'Full-Volume Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_multi_view(
    input_vol, pred_masked, target_vol,
    title=f"{test_subject['subject']} - Step {step_loaded} | SSIM={ssim_masked:.4f} PSNR={psnr_masked:.2f}dB"
)

In [ ]:
# ============================================================
# Error Distribution Analysis
# ============================================================

brain_errors = np.abs(pred_vol[brain_mask] - target_vol[brain_mask])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Error histogram
axes[0].hist(brain_errors, bins=100, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].axvline(x=brain_errors.mean(), color='red', linestyle='--', label=f'Mean={brain_errors.mean():.4f}')
axes[0].axvline(x=np.median(brain_errors), color='orange', linestyle='--', label=f'Median={np.median(brain_errors):.4f}')
axes[0].set_title('Error Distribution (Brain Region)', fontsize=13)
axes[0].set_xlabel('Absolute Error')
axes[0].set_ylabel('Voxel Count')
axes[0].legend()

# Intensity comparison
sample_idx = np.random.choice(brain_mask.sum(), size=min(50000, brain_mask.sum()), replace=False)
pred_sample = pred_vol[brain_mask][sample_idx]
tgt_sample = target_vol[brain_mask][sample_idx]
axes[1].scatter(tgt_sample, pred_sample, s=0.5, alpha=0.2, color='steelblue')
axes[1].plot([-1, 1], [-1, 1], 'r--', linewidth=1)
axes[1].set_title('Predicted vs Target Intensity', fontsize=13)
axes[1].set_xlabel('Target 7T')
axes[1].set_ylabel('Predicted 7T')
axes[1].set_xlim(-1, 1)
axes[1].set_ylim(-1, 1)

# Per-slice SSIM
slice_ssim = []
for s in range(D):
    if brain_mask[s].any():
        p_sl = (pred_masked[s] + 1) / 2
        t_sl = (target_vol[s] + 1) / 2
        try:
            slice_ssim.append(ssim(t_sl, p_sl, data_range=1.0))
        except:
            slice_ssim.append(np.nan)
    else:
        slice_ssim.append(np.nan)

axes[2].plot(range(D), slice_ssim, 'b-', alpha=0.7)
axes[2].axhline(y=ssim_masked, color='red', linestyle='--', label=f'Volume SSIM={ssim_masked:.4f}')
axes[2].set_title('Per-Slice SSIM (Axial)', fontsize=13)
axes[2].set_xlabel('Slice Index')
axes[2].set_ylabel('SSIM')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Error Analysis - {test_subject["subject"]}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Error stats (brain region):")
print(f"  Mean: {brain_errors.mean():.4f}")
print(f"  Median: {np.median(brain_errors):.4f}")
print(f"  95th percentile: {np.percentile(brain_errors, 95):.4f}")
print(f"  Max: {brain_errors.max():.4f}")

---
## 8. Cross-Subject Evaluation

Evaluate the model across all subjects (or test subjects) to get aggregate metrics.

In [ ]:
# ============================================================
# Evaluate All Subjects
# ============================================================

# Select which subjects to evaluate
# Option 1: All subjects
eval_pairs = pairs
# Option 2: Only test subject(s) - uncomment below
# splitter = SubjectSplitter(split_cfg)
# _, _, test_pairs = splitter.get_split(pairs)
# eval_pairs = test_pairs

all_metrics = []

for pair in tqdm(eval_pairs, desc="Evaluating subjects"):
    subj = pair['subject']
    inp_path = pair['input_3t']
    tgt_path = pair['target_7t']
    
    if not Path(inp_path).exists() or not Path(tgt_path).exists():
        print(f"Skipping {subj}: files not found")
        continue
    
    # Load volumes
    inp_vol = np.clip(nib.load(inp_path).get_fdata().astype(np.float32), -1, 1)
    tgt_vol = np.clip(nib.load(tgt_path).get_fdata().astype(np.float32), -1, 1)
    
    # Run tiled inference
    pred, seg = tiled_inference(inf_diffusion, inp_vol, device, patch_size=64, overlap=32)
    
    # Brain mask
    bmask = inp_vol > -0.95
    pred_m = pred.copy()
    pred_m[~bmask] = tgt_vol[~bmask]
    
    # Metrics
    s_masked, p_masked = compute_metrics(pred_m, tgt_vol)
    l1_val, l2_val = compute_l1_l2(pred_m[bmask], tgt_vol[bmask])
    
    # Brain bounding box
    coords = np.argwhere(bmask)
    lo = coords.min(axis=0)
    hi = coords.max(axis=0) + 1
    s_bb, p_bb = compute_metrics(
        pred_m[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]],
        tgt_vol[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    )
    
    metrics = {
        'subject': subj,
        'ssim_masked': s_masked,
        'psnr_masked': p_masked,
        'ssim_brain': s_bb,
        'psnr_brain': p_bb,
        'l1': l1_val,
        'l2': l2_val,
    }
    all_metrics.append(metrics)
    print(f"  {subj}: SSIM={s_masked:.4f}, PSNR={p_masked:.2f}dB")
    
    # Save individual result
    subj_dir = OUTPUT_DIR / 'results' / subj
    subj_dir.mkdir(parents=True, exist_ok=True)
    nib.save(
        nib.Nifti1Image(pred, nib.load(inp_path).affine),
        subj_dir / 'predicted_7T.nii.gz'
    )

print(f"\nEvaluated {len(all_metrics)} subjects.")

In [ ]:
# ============================================================
# Aggregate Results Table
# ============================================================
import pandas as pd

if all_metrics:
    df = pd.DataFrame(all_metrics)
    
    print("\n" + "="*70)
    print("CROSS-SUBJECT EVALUATION RESULTS")
    print(f"Checkpoint: step {step_loaded}")
    print("="*70)
    
    display(df.round(4))
    
    # Summary statistics
    print("\nSummary (mean +/- std):")
    for col in ['ssim_masked', 'psnr_masked', 'ssim_brain', 'psnr_brain', 'l1', 'l2']:
        mean = df[col].mean()
        std = df[col].std()
        print(f"  {col:<15}: {mean:.4f} +/- {std:.4f}")
    
    # Save to CSV
    csv_path = OUTPUT_DIR / 'results' / f'metrics_step{step_loaded}.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nMetrics saved to: {csv_path}")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].bar(df['subject'], df['ssim_masked'], color='steelblue', alpha=0.8)
    axes[0].axhline(y=df['ssim_masked'].mean(), color='red', linestyle='--',
                    label=f'Mean={df["ssim_masked"].mean():.4f}')
    axes[0].set_title('SSIM per Subject', fontsize=13)
    axes[0].set_ylabel('SSIM')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=45)
    
    axes[1].bar(df['subject'], df['psnr_masked'], color='darkorange', alpha=0.8)
    axes[1].axhline(y=df['psnr_masked'].mean(), color='red', linestyle='--',
                    label=f'Mean={df["psnr_masked"].mean():.2f}')
    axes[1].set_title('PSNR per Subject', fontsize=13)
    axes[1].set_ylabel('PSNR (dB)')
    axes[1].legend()
    axes[1].tick_params(axis='x', rotation=45)
    
    axes[2].bar(df['subject'], df['l1'], color='seagreen', alpha=0.8)
    axes[2].axhline(y=df['l1'].mean(), color='red', linestyle='--',
                    label=f'Mean={df["l1"].mean():.4f}')
    axes[2].set_title('L1 Error per Subject', fontsize=13)
    axes[2].set_ylabel('L1')
    axes[2].legend()
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.suptitle(f'Cross-Subject Metrics (Step {step_loaded})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'results' / f'metrics_step{step_loaded}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No metrics collected. Check that data files exist.")

---
## 9. Inference on New Data (No Ground Truth)

Use this section to run inference on new 3T volumes where no 7T ground truth exists.

In [ ]:
# ============================================================
# Inference on a New 3T Volume (No Ground Truth)
# ============================================================

# Set path to your new 3T volume (must be preprocessed & normalized to [-1, 1])
NEW_INPUT_PATH = None  # e.g., Path(r"D:\data\new_subject\T1w_preproc.nii.gz")
NEW_OUTPUT_DIR = OUTPUT_DIR / 'results' / 'new_subject'

if NEW_INPUT_PATH and Path(NEW_INPUT_PATH).exists():
    print(f"Loading: {NEW_INPUT_PATH}")
    new_nii = nib.load(str(NEW_INPUT_PATH))
    new_vol = np.clip(new_nii.get_fdata().astype(np.float32), -1.0, 1.0)
    
    print(f"Volume shape: {new_vol.shape}")
    print(f"Range: [{new_vol.min():.3f}, {new_vol.max():.3f}]")
    
    # Run tiled inference
    print("Running inference...")
    new_pred, new_seg = tiled_inference(inf_diffusion, new_vol, device, patch_size=64, overlap=32)
    
    # Apply brain mask
    new_brain_mask = new_vol > -0.95
    new_pred_masked = new_pred.copy()
    new_pred_masked[~new_brain_mask] = new_vol[~new_brain_mask]
    
    # Save
    NEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    nib.save(nib.Nifti1Image(new_pred_masked, new_nii.affine), NEW_OUTPUT_DIR / 'predicted_7T.nii.gz')
    nib.save(nib.Nifti1Image(new_seg.astype(np.float32), new_nii.affine), NEW_OUTPUT_DIR / 'predicted_seg.nii.gz')
    
    print(f"Saved to: {NEW_OUTPUT_DIR}")
    
    # Visualize
    D, H, W = new_vol.shape
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    sl = D // 2
    axes[0].imshow(np.rot90(new_vol[sl]), cmap='gray', vmin=-1, vmax=1)
    axes[0].set_title('3T Input', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(np.rot90(new_pred_masked[sl]), cmap='gray', vmin=-1, vmax=1)
    axes[1].set_title('Predicted 7T', fontsize=14)
    axes[1].axis('off')
    axes[2].imshow(np.rot90(new_seg[sl]), cmap='viridis', vmin=0, vmax=3)
    axes[2].set_title('Segmentation', fontsize=14)
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Set NEW_INPUT_PATH to run inference on new data.")
    print("The input must be preprocessed and normalized to [-1, 1].")

---
## Summary

This notebook provides the complete Topo-Brain pipeline:

| Section | Purpose |
|---------|--------|
| 1. Setup | Load config, verify GPU & imports |
| 2. Data Exploration | Visualize 3T/7T pairs and patches |
| 3. Training | Full curriculum training with all losses |
| 4. Checkpoints | List and select best checkpoint |
| 5. Patch Inference | Quick 64^3 center crop inference |
| 6. Full-Volume | Tiled inference with Tukey blending |
| 7. Evaluation | SSIM, PSNR, L1, L2, segmentation |
| 8. Cross-Subject | Aggregate metrics across subjects |
| 9. New Data | Inference without ground truth |

### Key Notes
- All data must be preprocessed and normalized to [-1, 1] before use
- Training uses 4-stage curriculum for stable convergence
- EMA weights are used for inference (more stable)
- Tiled inference with 50% overlap prevents boundary artifacts
- Metrics are computed on brain-masked regions only